In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, HBox, Layout, VBox, HTML
from IPython.display import display

def plot_lp_filter_specs(delta_p=0.08, delta_s=0.05, wp=1.0, ws=1.8):
    fig, ax = plt.subplots(figsize=(10, 6))
    omega = np.linspace(0.0, np.pi, 1000)

    H_mag = np.zeros_like(omega)

    for i, w in enumerate(omega):
        if w <= wp:
            ratio = w / wp if wp > 0 else 0.0
            ripple = np.sin(4.0 * np.pi * ratio)
            H_mag[i] = 1.0 + delta_p * ripple

        elif wp < w < ws:
            ratio = (w - wp) / (ws - wp)
            val_wp = 1.0 - delta_p
            val_ws = delta_s
            H_mag[i] = val_wp + ratio * (val_ws - val_wp)

        else:
            if ws < np.pi:
                ratio = (w - ws) / (np.pi - ws)
                ripple = 0.5 * (1.0 + np.cos(5.0 * np.pi * ratio))
                H_mag[i] = delta_s * ripple
            else:
                H_mag[i] = 0.0

    upper_bound_pb = 1.0 + delta_p
    lower_bound_pb = 1.0 - delta_p
    stop_bound = delta_s

    ax.plot(omega, H_mag, 'r-', linewidth=2, label=r'$|H(e^{j\omega})|$')

    ax.hlines(upper_bound_pb, 0, wp, colors='k', linewidth=1.5)

    ax.hlines(lower_bound_pb, 0, wp, colors='k', linewidth=1.5, linestyle='--')

    ax.hlines(stop_bound, ws, np.pi, colors='k', linewidth=1.5)

    ax.axvline(wp, ymin=0, ymax=0.8, color='k', linestyle='--', linewidth=1)

    ax.axvline(ws, ymin=0, ymax=0.35, color='k', linestyle='--', linewidth=1)

    ax.fill_between([0, wp], upper_bound_pb, 1.25, color='blue', alpha=0.1, hatch='//')

    ax.fill_between([0, wp], 0.0, lower_bound_pb, color='blue', alpha=0.1, hatch='//')

    ax.fill_between([ws, np.pi], stop_bound, 1.0, color='blue', alpha=0.1, hatch='//')

    ax.set_xlim(0, np.pi)

    ax.set_ylim(-0.02, 1.25)

    ax.set_xlabel(r'$\omega$', fontsize=14)

    ax.set_ylabel(r'$|H(e^{j\omega})|$', fontsize=14)

    ax.set_xticks([wp, ws, np.pi])

    ax.set_xticklabels([r'$\omega_p$', r'$\omega_s$', r'$\pi$'], fontsize=12)

    y_ticks = [stop_bound, lower_bound_pb, 1.0, upper_bound_pb]

    y_labels = [r'$\delta_s$', r'$1-\delta_p$', '1', r'$1+\delta_p$']

    ax.set_yticks(y_ticks)

    ax.set_yticklabels(y_labels, fontsize=12)

    ax.text(wp / 2, 0.5, 'Passband', color='green', fontsize=11, fontweight='bold', ha='center')

    ax.text((wp + ws) / 2, 0.5, 'Transition Band', color='green', fontsize=10, fontweight='bold', ha='center')

    ax.text(ws + (np.pi - ws) / 2, 0.5, 'Stopband', color='green', fontsize=11, fontweight='bold', ha='center')

    ax.grid(True, linestyle=':', alpha=0.6)

    plt.title('Normalized Frequency Response of a Low-Pass Digital Filter', fontsize=13, pad=15)

    plt.show()


slider_layout = Layout(width='310px')

style_opts = {'description_width': '55px'}

dp_slider = FloatSlider(min=0.01, max=0.20, step=0.01, value=0.08, description='δp:', style=style_opts, layout=slider_layout)

ds_slider = FloatSlider(min=0.01, max=0.20, step=0.01, value=0.05, description='δs:', style=style_opts, layout=slider_layout)

wp_slider = FloatSlider(min=0.2, max=1.5, step=0.05, value=1.0, description='ωp:', style=style_opts, layout=slider_layout)

ws_slider = FloatSlider(min=1.55, max=3.0, step=0.05, value=1.8, description='ωs:', style=style_opts, layout=slider_layout)

widget_plot = interactive(plot_lp_filter_specs, delta_p=dp_slider, delta_s=ds_slider, wp=wp_slider, ws=ws_slider)

theory_html = HTML("""
<div style="font-family: monospace; font-size: 13px; line-height: 1.5; margin-bottom: 8px;">
<b>δp:</b> Maximum allowed passband deviation.<br>
<b>δs:</b> Maximum allowed stopband magnitude.<br>
<b>ωp:</b> Passband edge frequency.<br>
<b>ωs:</b> Stopband edge frequency.
</div>
""")

controls = VBox([dp_slider, ds_slider, wp_slider, ws_slider], layout=Layout(width='340px', min_width='340px', overflow='visible', justify_content='center'))

main_layout = HBox([widget_plot.children[-1], controls], layout=Layout(width='100%', overflow='visible', align_items='center', justify_content='flex-start'))

display(VBox([theory_html, main_layout], layout=Layout(width='100%', overflow='visible', align_items='flex-start')))